# TP2 - Sistemas Inteligentes: Recomendador grupal de películas y libros

**Caso de negocio:** un grupo de 5 amigos nunca se pone de acuerdo en qué ver o leer. Este sistema cruza los 5 perfiles de gustos, busca candidatos en una base de sinopsis (RAG) y devuelve UNA recomendación final justificada, en tono canchero argentino.

**Arquitectura:** `ingestion/` (TMDB + Google Books) -> `data/dataset.csv` -> `rag/` (embeddings + ChromaDB) -> `agent/` (cruza perfiles, RAG, rankea) -> `prompts/` (personalidad) -> respuesta final. Ver `docs/DISENO.md` en el repo para el detalle de cada decisión.

**Cómo correr esta notebook:**
1. Subir/clonar el repo completo a Colab (celda siguiente).
2. Cargar los secrets (`GOOGLE_API_KEY`, `TMDB_API_KEY`) en el panel de secrets de Colab (ícono de llave a la izquierda) con esos mismos nombres.
3. Correr las celdas en orden.

## 1. Setup del entorno

In [1]:
# Si estás en Colab, clonar el repo
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    !git clone https://github.com/JuanBona/EntregaTPI_IA.git repo
    %cd repo
    !pip install -q -r requirements.txt

In [2]:
# Cargar API keys: en Colab desde userdata (Secrets), en local desde .env
import os

if IN_COLAB:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    try:
        os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")
    except Exception:
        pass  # no hace falta salvo que REGENERAR_DATASET sea True (celda de abajo)
    os.environ.setdefault("GEMINI_MODEL", "gemini-3.6-flash")
    os.environ.setdefault("CHROMA_PERSIST_DIR", "./data/chroma")
    os.environ.setdefault("DATASET_CSV_PATH", "./data/dataset.csv")
else:
    from dotenv import load_dotenv

    load_dotenv()

## 2. Ingesta de datos (`/ingestion`)

**Decisión de diseño:** si `data/dataset.csv` ya está commiteado en el repo, no hace
falta correr la ingesta de nuevo (ahorra tiempo y evita gastar la cuota de las APIs
el día de la defensa). Solo correr la celda de abajo si querés refrescar el dataset.

In [3]:
REGENERAR_DATASET = False  # poner True para volver a pegarle a TMDB/Google Books

if REGENERAR_DATASET:
    !python -m ingestion.build_dataset

import pandas as pd

df = pd.read_csv(os.environ["DATASET_CSV_PATH"])
print(f"Dataset: {len(df)} filas")
df.head()

Dataset: 276 filas


,id,tipo,titulo,sinopsis,generos,rating,anio,fuente,fuente_id,autor_director,idioma
0,016d6b3ab0b8182a,pelicula,Spider-Man: Brand New Day,Han pasado cuatro años desde los acontecimient...,Ciencia ficción|Acción|Aventura,7.865,2026,tmdb,969681,NaN,en
1,349df56f205bbd4d,pelicula,Coyote vs. Acme,Después de que todos los productos fabricados ...,Comedia|Aventura|Familia,7.700,2026,tmdb,1204680,NaN,en
2,03dcb62e624909d7,pelicula,La Odisea,"Tras la caída de Troya, el legendario rey de Í...",Aventura|Acción|Fantasía,7.999,2026,tmdb,1368337,NaN,en
3,4e6788c80cb297e0,pelicula,El motín,"Después de que su jefe, un industrial multimil...",Acción|Suspense,6.400,2026,tmdb,1288445,NaN,en
4,186dfecd032c4221,pelicula,A la carrera,"En el thriller psicológico A la carrera, Maia ...",Suspense|Acción,6.736,2026,tmdb,1386315,NaN,en


## 3. RAG: embeddings + ChromaDB (`/rag`)

**Decisión de diseño:** embeddings locales multilingües de HuggingFace en vez de
OpenAI, para no sumar una dependencia/costo extra al stack de Gemini. Ver
`docs/DISENO.md` para el trade-off completo.

In [4]:
from rag.vectorstore import construir_vectorstore

vectorstore = construir_vectorstore()
print(f"Vectorstore listo con {vectorstore._collection.count()} documentos.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vectorstore listo con 276 documentos.


In [5]:
# Prueba rápida del retriever
from rag.retriever import retriever

retriever("thriller psicológico con giros de guión", k=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'titulo': 'Insidious: Fuera del más allá',
  'tipo': 'pelicula',
  'generos': 'Terror|Suspense',
  'rating': 6.483,
  'anio': 2026,
  'autor_director': None,
  'sinopsis': 'La saga de terror vuelve a sus raíces sobrenaturales. Una nueva protagonista se enfrenta a misteriosas apariciones que van mucho más allá de los fenómenos de embrujamiento clásicos. El mal encuentra nuevos caminos en la realidad. Un trío de acechadores se infiltra en un tranquilo suburbio obligando a una nueva familia a entrar en el plano astral, donde descubren una aterradora verdad: el Más Allá está sangrando en el mundo real',
  'score': 3.3762810230255127},
 {'titulo': 'Blood Covenant',
  'tipo': 'pelicula',
  'generos': 'Terror',
  'rating': 5.5,
  'anio': 2026,
  'autor_director': None,
  'sinopsis': 'Sigue a un prometedor escritor de terror en el momento más bajo de su vida: años sin éxito, ahogado por las deudas y al borde del colapso. Desesperado, lleva a cabo un ritual ocultista para invocar a un demonio

## 4. Perfiles del grupo (`/perfiles`)

Cada integrante completó su `perfiles/<nombre>.json` (ver `perfiles/README.md` para el
esquema). Se cargan todos automáticamente.

In [6]:
from agent.perfil import cargar_todos_los_perfiles, validar_generos_contra_dataset

perfiles = cargar_todos_los_perfiles()
for p in perfiles:
    print(p.nombre)

# Chequeo rapido: generos que no existen en el dataset y por eso nunca van a matchear
validar_generos_contra_dataset(perfiles)

Bruno
Juan
Lucas
Marcos
Rafa
ojo: el género 'musical' de Bruno no existe en el dataset
ojo: el género 'musical' de Lucas no existe en el dataset
ojo: el género 'policial' de Rafa no existe en el dataset
ojo: el género 'policial' de Rafa no existe en el dataset
ojo: el género 'ensayo' de Rafa no existe en el dataset


## 5. Agente mediador (`/agent` + `/prompts`)

Cruza los 5 perfiles, busca candidatos vía RAG, los rankea por afinidad grupal y le
pide a Gemini la recomendación final con el tono de `prompts/system_prompt.py`.

In [7]:
from agent.mediador import recomendar_grupal

resultado = recomendar_grupal(perfiles)
print("Criterio de búsqueda usado:", resultado["criterio_busqueda"])
print()
print("=== Recomendación final ===")
print(resultado["recomendacion"])

C:\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Criterio de búsqueda usado: terror, thriller, ciencia ficcion, terror, misterio, Hereditary, It, Get Out, accion, aventura, ciencia ficcion, fantasia, aventura, Mad Max: Fury Road, The Hobbit, Arrival, ciencia ficcion, accion, thriller, ciencia ficcion, historia, Interstellar, Dune, The Bourne Identity, comedia, drama, biografia, biografia, ficcion, The Grand Budapest Hotel, Sapiens, Little Miss Sunshine, drama, policial, misterio, policial, ensayo, Prisoners, The Silent Patient, Zodiac

=== Recomendación final ===
Che, estuve mirando lo que hay disponible y las ganas de cada uno, y aunque la mesa está variada, hay una opción que es la que mejor equilibra los tantos.

La recomendación principal para ver hoy es la película **"Above & Below"**.

**¿Por qué le cierra al grupo?**
La historia arranca con un grupo de amigos de vacaciones, a puro buceo y fiesta, pero todo se pudre mal cuando los atacan unos criminales en mar abierto y pasa a ser una pesadilla de supervivencia. 

A **Juan** y 

In [8]:
# Candidatos que consideró el agente, para mostrar en la defensa cómo llegó a la respuesta
import pandas as pd

pd.DataFrame(resultado["candidatos_rankeados"])

,titulo,tipo,generos,rating,anio,autor_director,sinopsis,score,score_grupal
0,Vengadores: Doomsday,pelicula,Ciencia ficción|Acción|Aventura,0.000,2026,None,Quinta entrega de la saga 'Vengadores' incluid...,7.047129,6.0000
1,Above & Below,pelicula,Acción|Suspense|Terror|Crimen,6.200,2026,None,Un grupo de amigos se embarca en unas vacacion...,6.769502,3.6200
2,Insidious: Fuera del más allá,pelicula,Terror|Suspense,6.483,2026,None,La saga de terror vuelve a sus raíces sobrenat...,6.588389,1.6483


## 6. Casos de prueba para la defensa

Documentar acá 2-3 corridas con distintos perfiles/grupos para mostrar que el sistema
funciona con casos variados (pedido explícito de la rúbrica: "Funcionamiento del
sistema" y "Presentación y defensa").

### Caso 1 — grupo completo (los 5 perfiles reales de `perfiles/`)

Corrida real de `recomendar_grupal()` con los perfiles de Bruno, Juan, Lucas, Marcos
y Rafa tal como estan en el repo (ver el output de la celda de arriba). El sistema
recomendo **"Above & Below"** (thriller de supervivencia), explicando por que le
cierra a cada uno y por que descarto *Insidious* (Marcos y Rafa no bancan terror
paranormal) y *Vengadores: Doomsday* (sin sinopsis ni rating confirmado en el
dataset). Se corrio dos veces con el mismo grupo y en ambas eligio la misma
pelicula, cambiando solo la redaccion — buena senal de consistencia.

**Limitacion visible en la tabla de candidatos:** `autor_director` sale vacio para
las 3 peliculas candidatas. Es un problema conocido de `ingestion/` (TMDB no trae el
director sin un request extra por pelicula) — limitacion asumida, documentada en
`docs/DISENO.md`.

### Caso 2 — el retriever distingue una query off-topic

Sin gastar cuota del LLM: probando `rag.retriever()` solo, una query clara
("aventura espacial con toques de humor") trae candidatos con distancia semantica
3.7-6.5, mientras que una query totalmente ajena al dataset ("recetas de cocina
italiana") trae distancia 10.0-10.9 — casi el doble. Hay margen claro para un
umbral anti-alucinacion (ej. "si el mejor candidato tiene distancia > 9, decir que
no se encontro nada que le cierre al grupo" en vez de inventar una recomendacion).
Resultados completos en `p12_checklist_resultados.txt`.

### Caso 3 — una query clara de libro no trajo ni un libro

Probando `rag.retriever("novela policial nordica")`: los 5 resultados fueron todas
peliculas de accion/crimen, ningun libro. Coincide con lo que ya sabiamos por P2: en
el dataset las peliculas policiales estan etiquetadas `Crimen` y no `policial`, asi
que el embedding tira para el lado de accion/crimen en general en vez de narrativa
policial de libro. Vale la pena mencionarlo en la seccion de dificultades de
`DISENO.md` junto con P2 y P3 — es el mismo tipo de problema (vocabulario que no
matchea), encontrado en un lugar distinto del sistema.

*Nota: se documentaron 3 casos con cuota limitada de la API gratuita de Gemini (20
requests/dia). Si el equipo quiere probar con perfiles distintos a los del repo,
alcanza con pasarle una lista de `Perfil` a `recomendar_grupal(perfiles)` en vez de
dejar que cargue los de `perfiles/` por default.*